# DSPy — Otimização com `MIPROv2`

Neste notebook será demonstrado o uso do otimizador `MIPROv2` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy utilizando a instrução original definida na `Signature`;
2. utilizar o `MIPROv2` para otimizar conjuntamente a **instrução** e as **demonstrações few-shot** utilizadas pelo programa.

`MIPROv2` significa **Multiprompt Instruction PRoposal Optimizer Version 2**.

O otimizador trabalha, em alto nível, em três etapas:

1. cria candidatos de demonstrações few-shot a partir do conjunto de treinamento;
2. propõe diferentes instruções levando em consideração a tarefa, os dados e as demonstrações disponíveis;
3. utiliza **otimização Bayesiana** para buscar a melhor combinação entre instrução e exemplos few-shot.

Neste experimento, os dados serão separados em:

* **treino**, utilizado para gerar candidatos de demonstrações e auxiliar a proposta de instruções;
* **validação**, utilizada para comparar combinações candidatas e selecionar o programa final;
* **teste**, mantido separado para a avaliação final.

A avaliação final utilizará:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica principal para comparar o programa original e o programa otimizado será o **F1-score**.


In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

Serão utilizados dois papéis de modelo no `MIPROv2`:

* `lm`: executa o classificador e funciona como `task_model`;
* `prompt_lm`: é utilizado como `prompt_model` para analisar a tarefa e gerar novas instruções candidatas.

Neste exemplo, ambos utilizarão `openai/gpt-5-mini`, mas poderiam ser modelos diferentes.

O `MIPROv2` utiliza o **Optuna** para a etapa de otimização Bayesiana. Caso a dependência opcional ainda não esteja instalada no ambiente com `uv`, pode-se executar:

```bash
uv add "dspy[optuna]"
```


In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo utilizado para executar o classificador
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Modelo utilizado pelo MIPROv2 para gerar instruções candidatas.
# O init_temperature do MIPROv2 será mantido em 1.0, compatível com modelos GPT-5.
prompt_lm = dspy.LM(
    "openai/gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
)

# Configura o modelo do classificador como LM padrão do DSPy
dspy.configure(lm=lm)


## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino, validação e teste

Para utilizar o `MIPROv2` de forma adequada, a base será dividida em três conjuntos:

* **70% para treino**;
* **15% para validação**;
* **15% para teste**.

O parâmetro `stratify` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` em todos os conjuntos.

No `MIPROv2`, os conjuntos possuem funções diferentes:

* o **trainset** fornece exemplos para o bootstrap de demonstrações few-shot e também informações sobre a tarefa utilizadas na proposta de novas instruções;
* o **valset** é utilizado para avaliar as diferentes combinações de instruções e demonstrações durante a busca;
* o **testset** não participa da otimização e é utilizado somente na comparação final entre o baseline e o programa otimizado.

Essa separação evita utilizar o conjunto de teste para selecionar os prompts.


In [7]:
# Primeiro separamos 70% para treino e 30% para validação + teste
df_train, df_temp = train_test_split(
    df[["text", "target"]],
    test_size=0.30,
    random_state=42,
    stratify=df["target"],
)

# Divide os 30% restantes igualmente: 15% validação e 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     140 exemplos
Validação:  30 exemplos
Teste:      30 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Valset DSPy:   {len(valset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 140
Valset DSPy:   30
Testset DSPy:  30


In [10]:
trainset[0]

Example({'text': '13,000 people receive #wildfires evacuation orders in California ', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Classifique o tweet em uma das duas classes possíveis.
    """

    text: str = dspy.InputField(
        desc="Tweet a ser analisado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="Classe prevista."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|█| 30/30 [03:5


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.8966


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9000
Precision: 0.9286
Recall:    0.8667
F1:        0.8966


## Métrica utilizada durante a otimização

O `MIPROv2` precisa de uma métrica aplicada a cada exemplo para avaliar os programas candidatos.

Neste problema será utilizada uma métrica simples de **acerto da classe**:

```python
def metrica_mipro(example, prediction, trace=None):
    return int(example.target) == int(prediction.target)
```

A métrica retorna:

* `True` quando a classe prevista coincide com a classe esperada;
* `False` quando a classificação está incorreta.

Essa métrica é apropriada para orientar o bootstrap e a busca do `MIPROv2`, pois cada execução é avaliada individualmente.

O **F1-score global** continuará sendo calculado separadamente sobre todo o conjunto de teste. Isso é importante porque F1 é uma métrica agregada e não deve ser interpretada a partir de um único exemplo isolado.

Diferentemente do `GEPA`, não precisamos retornar feedback textual para orientar reflexão.


In [17]:
def metrica_mipro(example, prediction, trace=None):
    """
    Métrica utilizada internamente pelo MIPROv2.

    Retorna True quando a classe prevista é igual à classe esperada
    e False caso contrário.
    """
    esperado = int(example.target)
    previsto = int(prediction.target)

    return esperado == previsto


## Otimização com `MIPROv2`

O `MIPROv2` (**Multiprompt Instruction PRoposal Optimizer Version 2**) otimiza prompts combinando **instruções candidatas** e **demonstrações few-shot**.

Neste notebook, o programa possui apenas um `dspy.Predict`. Para esse preditor, o MIPROv2 poderá alterar:

* a instrução da `Signature`;
* o conjunto de demonstrações few-shot inserido no programa.

O funcionamento pode ser representado conceitualmente como:

```text
programa original
       ↓
bootstrap de exemplos do trainset
       ↓
conjuntos candidatos de demonstrações few-shot
       ↓
análise da tarefa e dos dados
       ↓
geração de instruções candidatas
       ↓
combinações: instrução + demos
       ↓
avaliação no valset
       ↓
otimização Bayesiana
       ↓
melhor combinação encontrada
```

A otimização Bayesiana evita testar todas as combinações possíveis de forma exaustiva. Em vez disso, utiliza os resultados dos trials anteriores para decidir quais combinações parecem mais promissoras.

Com `auto="light"`, o DSPy configura automaticamente o orçamento da busca, incluindo a quantidade de candidatos e trials.


In [18]:
AUTO = "light"

optimizer = dspy.MIPROv2(
    metric=metrica_mipro,          # Métrica usada para avaliar cada previsão
    prompt_model=prompt_lm,        # LM que propõe novas instruções
    task_model=lm,                 # LM que executa os programas candidatos
    auto=AUTO,                     # Orçamento automático: "light", "medium" ou "heavy"
    max_bootstrapped_demos=4,      # Máximo de demos produzidas por bootstrap
    max_labeled_demos=4,           # Máximo de demos rotuladas diretamente do trainset
    num_threads=4,                 # Paralelismo das avaliações
    init_temperature=1.0,          # Temperatura usada na geração de instruções candidatas
    seed=42,                       # Reprodutibilidade
    verbose=False,                 # True mostra detalhes adicionais da geração de candidatos
    track_stats=True,              # Mantém logs e programas candidatos para inspeção
)


## Compilação do programa

A compilação do `MIPROv2` recebe:

* o programa que será otimizado (`student`);
* o `trainset`, utilizado para criar demonstrações e orientar a proposta de instruções;
* o `valset`, utilizado para comparar os programas candidatos.

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)
```

Como `auto` está definido como `"light"`, não devemos informar manualmente `num_candidates` ou `num_trials`: esses valores são configurados automaticamente pelo otimizador.

Neste experimento pequeno, o conjunto de validação possui menos de 50 exemplos. Na implementação atual, o modo automático desativa o uso de minibatches nesse caso e avalia os candidatos diretamente no conjunto de validação selecionado.


In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)

2026/09/07 23:18:19 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 30

2026/09/07 23:18:19 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/09/07 23:18:19 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/09/07 23:18:19 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


  4%|  | 5/140 [00:58<26:18, 11.69s/it]


Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
Bootstrapping set 4/6


  1%|  | 2/140 [00:16<18:29,  8.04s/it]


...

2026/09/07 23:28:05 INFO dspy.evaluate.evaluate: Average Metric: 29 / 30 (96.7%)
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 96.67 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 3'].
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [83.33, 93.33, 96.67, 96.67, 93.33, 96.67, 96.67, 93.33, 96.67]
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 96.67
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 10 / 10 =====



Average Metric: 28.00 / 30 (93.3%): 100

2026/09/07 23:28:05 INFO dspy.evaluate.evaluate: Average Metric: 28 / 30 (93.3%)
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 93.33 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 5'].
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [83.33, 93.33, 96.67, 96.67, 93.33, 96.67, 96.67, 93.33, 96.67, 93.33]
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 96.67
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====



Average Metric: 29.00 / 30 (96.7%): 100

2026/09/07 23:28:05 INFO dspy.evaluate.evaluate: Average Metric: 29 / 30 (96.7%)
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 96.67 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 0'].
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [83.33, 93.33, 96.67, 96.67, 93.33, 96.67, 96.67, 93.33, 96.67, 93.33, 96.67]
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 96.67
2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/09/07 23:28:05 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 96.67!


## Inspeção do resultado da otimização

Com `track_stats=True`, o programa retornado pelo `MIPROv2` recebe informações adicionais sobre a busca, como:

* `score`: melhor pontuação obtida na avaliação completa de validação;
* `trial_logs`: informações dos trials executados;
* `candidate_programs`: programas avaliados completamente, ordenados por score;
* `mb_candidate_programs`: candidatos avaliados em minibatches, quando minibatching é utilizado;
* `total_calls`: quantidade de chamadas contabilizadas durante a otimização;
* `prompt_model_total_calls`: chamadas realizadas pelo modelo responsável por propor prompts.

Também devemos inspecionar **duas partes** do programa final:

1. a instrução selecionada;
2. as demonstrações few-shot selecionadas.

Isso é importante porque o `MIPROv2` pode melhorar o programa mesmo que a instrução final permaneça igual à original, caso encontre um conjunto de demonstrações melhor.


In [20]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print("\n=== INSTRUÇÃO SELECIONADA PELO MIPROv2 ===")
print(classificador_otimizado.signature.instructions)

print("\n=== DEMONSTRAÇÕES FEW-SHOT SELECIONADAS ===")
print(f"Quantidade de demos: {len(classificador_otimizado.demos)}")

for i, demo in enumerate(classificador_otimizado.demos, start=1):
    print(f"\nDemo {i}:")
    print(demo)


=== INSTRUÇÃO ORIGINAL ===
Classifique o tweet em uma das duas classes possíveis.

=== INSTRUÇÃO SELECIONADA PELO MIPROv2 ===
Você é um classificador de tweets. Recebe um único texto curto (tweet) e deve retornar APENAS um dígito: 1 se o tweet relata ou descreve um incidente/emergência do mundo real (por exemplo acidente de trânsito em curso, incêndio ativo, evacuação, feridos/mortes, explosão, tiroteio, resgate, alerta operacional); 0 caso contrário. Não escreva nada além do dígito 0 ou 1.

Regras para decidir:
- Marque 1 (incidente) quando o texto contiver evidência clara de um evento real e recente/ em curso: palavras-chave incidentais (ex.: accident, crash, fire, killed, injured, evacuation, shots fired, explosion, ambulance), contagens de vítimas, ordens de evacuação, fechamentos de via, locais/saídas/rodovias (Exit 31, I-77, nome de cidade), timestamps, palavras de urgência ou ALL‑CAPS (BREAKING), ou links/imagens que explicitamente reportem o evento.
- Marque 0 (não incidente) q

In [21]:
print(f"Melhor score de validação: {classificador_otimizado.score}")
print(f"Chamadas totais contabilizadas: {classificador_otimizado.total_calls}")
print(f"Chamadas do prompt_model: {classificador_otimizado.prompt_model_total_calls}")
print(f"Programas com avaliação completa: {len(classificador_otimizado.candidate_programs)}")
print(f"Programas avaliados em minibatches: {len(classificador_otimizado.mb_candidate_programs)}")


Melhor score de validação: 96.67
Chamadas totais contabilizadas: 0
Chamadas do prompt_model: 0
Programas com avaliação completa: 11
Programas avaliados em minibatches: 0


In [22]:
historico_mipro = pd.DataFrame(
    [
        {
            "candidato": i,
            "score_validacao": candidato["score"],
        }
        for i, candidato in enumerate(classificador_otimizado.candidate_programs)
    ]
).sort_values(
    "score_validacao",
    ascending=False,
)

historico_mipro


,candidato,score_validacao
0,0,96.67
1,1,96.67
2,2,96.67
3,3,96.67
4,4,96.67
5,5,96.67
6,6,93.33
7,7,93.33
8,8,93.33
9,9,93.33


In [23]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="MIPROv2",
)


MIPROv2: 100%|█| 30/30 [03:41<00:00,  7


## Avaliação após a aplicação do `MIPROv2`

O programa otimizado pelo `MIPROv2` será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite comparar:

* o classificador utilizando a instrução original e sem demos selecionadas pelo otimizador;
* o classificador utilizando a combinação de instrução e demonstrações selecionada pelo `MIPROv2`.

O `testset` não participou da busca.

Durante a otimização:

* o `trainset` forneceu exemplos para o bootstrap de demonstrações e contexto para a geração de instruções;
* o `valset` foi utilizado para comparar as combinações candidatas;
* a otimização Bayesiana orientou quais combinações deveriam ser avaliadas ao longo dos trials.

Ao final serão calculados novamente:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica interna do `MIPROv2` é aplicada individualmente aos exemplos, enquanto o **F1-score global** continua sendo a métrica principal da comparação final no conjunto de teste.


In [24]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "MIPROv2",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao


,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.900000,0.928571,0.866667,0.896552
1,MIPROv2,0.966667,0.937500,1.000000,0.967742


In [25]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.8750    0.9333    0.9032        15
           1     0.9286    0.8667    0.8966        15

    accuracy                         0.9000        30
   macro avg     0.9018    0.9000    0.8999        30
weighted avg     0.9018    0.9000    0.8999        30



In [26]:
print(f"MIPROv2 (auto={AUTO})")

print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)


MIPROv2 (auto=light)
              precision    recall  f1-score   support

           0     1.0000    0.9333    0.9655        15
           1     0.9375    1.0000    0.9677        15

    accuracy                         0.9667        30
   macro avg     0.9688    0.9667    0.9666        30
weighted avg     0.9688    0.9667    0.9666        30



## Salvando o programa otimizado pelo `MIPROv2`

Neste experimento, o programa possui uma arquitetura simples baseada em `dspy.Predict`.

O `MIPROv2` pode modificar tanto a **instrução** quanto as **demonstrações few-shot** armazenadas no estado do preditor. Por isso, o **State-only Saving** em JSON continua sendo adequado:

```python
classificador_otimizado.save("MIPROv2.json")
```

Esse formato salva o estado otimizado do programa, incluindo os parâmetros necessários para reproduzir o prompt otimizado, mas não a definição Python completa da arquitetura.

Para carregar posteriormente, recriamos a mesma arquitetura (`dspy.Predict(ClassificarTweet)`) e aplicamos `.load()`.

Os atributos usados apenas para analisar o processo de busca, como `trial_logs` e `candidate_programs`, não são necessários para executar o classificador depois de carregado.


In [27]:
classificador_otimizado.save("MIPROv2.json")


In [28]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo MIPROv2
classificador_carregado.load("MIPROv2.json")

classificador_carregado


Predict(StringSignature(text -> target
    instructions='Você é um classificador de tweets. Recebe um único texto curto (tweet) e deve retornar APENAS um dígito: 1 se o tweet relata ou descreve um incidente/emergência do mundo real (por exemplo acidente de trânsito em curso, incêndio ativo, evacuação, feridos/mortes, explosão, tiroteio, resgate, alerta operacional); 0 caso contrário. Não escreva nada além do dígito 0 ou 1.\n\nRegras para decidir:\n- Marque 1 (incidente) quando o texto contiver evidência clara de um evento real e recente/ em curso: palavras-chave incidentais (ex.: accident, crash, fire, killed, injured, evacuation, shots fired, explosion, ambulance), contagens de vítimas, ordens de evacuação, fechamentos de via, locais/saídas/rodovias (Exit 31, I-77, nome de cidade), timestamps, palavras de urgência ou ALL‑CAPS (BREAKING), ou links/imagens que explicitamente reportem o evento.\n- Marque 0 (não incidente) quando o texto for: promoção/metáfora ("this song is fire", "kille

In [29]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [30]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T23:31:53.273573]

System message:

Your input fields are:
1. `text` (str): Tweet a ser analisado.
Your output fields are:
1. `target` (Literal[0, 1]): Classe prevista.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Você é um classificador de tweets. Recebe um único texto curto (tweet) e deve retornar APENAS um dígito: 1 se o tweet relata ou descreve um incidente/emergência do mundo real (por exemplo acidente de trânsito em curso, incêndio ativo, evacuação, feridos/mortes, explosão, tiroteio, resgate, alerta operacional); 0 caso contrário. Não escreva nada além do dígito 0 ou 1.
        
        Regras para decidir:
       